# 109 Python intermediate - menedżer kontekstu
_Kamil Bartocha_ | _wersja 2.0_

## Rozkład jazdy

1. 🔹 **Blok `with`** - po co, jak działa, zasoby
2. 🔹 **CM przez klasę** - `__enter__`, `__exit__`, obsługa wyjątków
3. 🔹 **@contextmanager** - generator jako CM
4. 🔹 **Praktyczne wzorce** - pliki, bazy danych, mierzenie czasu

🐍 Każda sekcja zawiera ćwiczenia.

## 1. 🔹 Blok `with`

Menedżer kontekstu (context manager) to mechanizm automatycznego
zarządzania zasobami - pliki, połączenia z bazą danych, blokady
wątków. Blok `with` gwarantuje wykonanie kodu "sprzątającego" nawet
gdy wystąpi wyjątek.

```python
# Bez with - ryzyko niezamkniętego pliku
f = open('file.txt')
data = f.read()      # co jesli tutaj wyjatek?
f.close()            # moze nie wykonac sie nigdy

# Z with - zamknięcie gwarantowane
with open('file.txt') as f:
    data = f.read()  # nawet przy wyjatku plik zostanie zamkniety
```

Można otwierać wiele zasobów naraz w jednym bloku `with`:

```python
with open('a.txt') as src, open('b.txt', 'w') as dst:
    dst.write(src.read())
```

> 💡 **Tip:** `with` to nie tylko pliki. Każdy obiekt z metodami
> `__enter__` i `__exit__` działa jako menedżer kontekstu:
> `threading.Lock()`, `sqlite3.connect()`, `requests.Session()` itd.

In [ ]:
# Zapis i odczyt pliku z with
with open('greetings.txt', 'w', encoding='utf-8') as f:
    f.write('Hello, Alice!\n')
    f.write('Hello, Bob!\n')

with open('greetings.txt', encoding='utf-8') as f:
    for line in f:
        print(line.strip())

# Dwa pliki naraz
with open('greetings.txt') as src, open('copy.txt', 'w') as dst:
    dst.write(src.read())

print('Done')

---

### 🐍 Ćwiczenia - Blok `with`

**Ćw. 1.** Użyj `with open()` żeby zapisać listę `lines` do pliku
`output.txt` (każda linia osobno), a następnie wczytaj i wyświetl
liczbę linii.

**Ćw. 2.** Napisz funkcję `copy_file(src, dst)`, która kopiuje
zawartość pliku `src` do `dst` używając bloku `with` z dwoma plikami.

**Ćw. 3. *(Trudniejsze)*** Napisz funkcję `word_count(path)`, która
wczytuje plik i zwraca słownik `{słowo: liczba_wystąpień}`.
Obsłuż `FileNotFoundError` zwracając pusty słownik.

In [ ]:
# Ćw. 1: zapis i odczyt
lines = ['apple', 'banana', 'cherry', 'date']

with open('output.txt', 'w') as f:
    ...

with open('output.txt') as f:
    content = ...

print(len(content))   # 4

In [ ]:
# Ćw. 2: copy_file
def copy_file(src, dst):
    with open(src) as s, open(dst, 'w') as d:
        ...


copy_file('output.txt', 'output_copy.txt')

with open('output_copy.txt') as f:
    print(f.read())

In [ ]:
# Ćw. 3: word_count
def word_count(path):
    counts = {}
    try:
        with open(path, encoding='utf-8') as f:
            for line in f:
                for word in line.split():
                    ...
    except FileNotFoundError:
        return {}
    return counts


print(word_count('greetings.txt'))   # {'Hello,': 2, 'Alice!': 1, 'Bob!': 1}
print(word_count('missing.txt'))     # {}

## 2. 🔹 CM przez klasę (`__enter__` i `__exit__`)

Własny menedżer kontekstu tworzymy przez klasę z dwiema metodami:

| Metoda | Kiedy | Co zwraca |
|--------|-------|----------|
| `__enter__(self)` | na początku bloku `with` | obiekt dostępny jako `as` |
| `__exit__(self, exc_type, exc_val, exc_tb)` | po bloku `with` | `True` = wyjątek stłumiony |

Parametry `__exit__`:
- `exc_type` - typ wyjątku (`None` gdy brak)
- `exc_val` - obiekt wyjątku
- `exc_tb` - traceback

Zwrócenie `True` z `__exit__` *tłumi* wyjątek (blok `with` kończy się
normalnie). Zwrócenie `False` lub `None` propaguje wyjątek dalej.

> 💡 **Tip:** Tłumienie wyjątków (`return True`) rzadko jest właściwe.
> Używaj go tylko gdy celowo chcesz zignorować określony typ błędu.

In [ ]:
# Podstawowy schemat klasy CM
class ManagedResource:
    def __enter__(self):
        print("Opening resource")
        return self          # dostepne jako 'as resource'

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type:
            print(f"Exception caught: {exc_type.__name__}: {exc_val}")
            return True      # stlum wyjatek
        print("Closing resource")
        return False


with ManagedResource() as res:
    print("Working...")

print("---")

with ManagedResource() as res:
    raise ValueError("something went wrong")

print("After block (exception was suppressed)")

In [ ]:
import time

# Praktyczny przyklad: mierzenie czasu bloku
class Timer:
    def __enter__(self):
        self.start = time.time()
        return self

    def __exit__(self, *args):
        self.elapsed = time.time() - self.start
        print(f"Elapsed: {self.elapsed:.4f}s")


with Timer() as t:
    total = sum(range(1_000_000))

print(f"Sum: {total}")
print(f"Time stored: {t.elapsed:.4f}s")

---

### 🐍 Ćwiczenia - CM przez klasę

**Ćw. 4.** Napisz klasę `Indenter`, która przy wchodzeniu zwiększa
wcięcie o 2 spacje i zmniejsza przy wyjściu. Udostępnia metodę
`print(text)` wyświetlającą tekst z bieżącym wcięciem.

**Ćw. 5.** Napisz klasę `SuppressError(*exc_types)`, która tłumi
podane typy wyjątków, a pozostałe propaguje dalej.

**Ćw. 6. *(Trudniejsze)*** Napisz klasę `TemporaryFile(filename)`,
która w `__enter__` tworzy plik i zwraca go, a w `__exit__`
zamyka i usuwa plik z dysku.

In [ ]:
# Ćw. 4: Indenter
class Indenter:
    def __init__(self):
        self.level = 0

    def __enter__(self):
        self.level += 1
        return self

    def __exit__(self, *args):
        self.level -= 1

    def print(self, text):
        ...


ind = Indenter()
with ind:
    ind.print('level 1')
    with ind:
        ind.print('level 2')
    ind.print('level 1 again')
# Expected:
#   level 1
#     level 2
#   level 1 again

In [ ]:
# Ćw. 5: SuppressError
class SuppressError:
    def __init__(self, *exc_types):
        self.exc_types = exc_types

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        # hint: sprawdz czy exc_type jest podklasa exc_types
        ...


with SuppressError(ValueError, KeyError):
    raise ValueError("ignored")   # stlumiony

print("After suppress block")

try:
    with SuppressError(ValueError):
        raise TypeError("not suppressed")
except TypeError as e:
    print(f"Got: {e}")

In [ ]:
import os

# Ćw. 6: TemporaryFile
class TemporaryFile:
    def __init__(self, filename):
        self.filename = filename
        self.file = None

    def __enter__(self):
        self.file = open(self.filename, 'w', encoding='utf-8')
        return self.file

    def __exit__(self, *args):
        # hint: zamknij plik i uzyj os.remove()
        ...


with TemporaryFile('tmp_test.txt') as f:
    f.write('temporary content')

print(os.path.exists('tmp_test.txt'))   # False

## 3. 🔹 `@contextmanager`

Dekorator `@contextmanager` z modułu `contextlib` pozwala pisać
menedżer kontekstu jako zwykłą funkcję generatora - bez klasy.
Wszystko *przed* `yield` to `__enter__`, wszystko *po* `yield`
to `__exit__`.

```python
from contextlib import contextmanager

@contextmanager
def my_cm():
    print("setup")       # __enter__
    yield resource        # wartość dostępna jako 'as'
    print("teardown")    # __exit__
```

Obsługa wyjątków - `try/finally` gwarantuje teardown:

```python
@contextmanager
def safe_cm():
    resource = acquire()
    try:
        yield resource
    finally:
        release(resource)  # wykonane zawsze
```

| Podejście | Zalety | Wady |
|-----------|--------|------|
| Klasa | jawne atrybuty, dziedziczenie | więcej kodu |
| `@contextmanager` | krótszy zapis, logika w jednym miejscu | trudniejsza obsługa wyjątków |

> 💡 **Tip:** Dla prostych przypadków użyj `@contextmanager`.
> Dla złożonych (wiele atrybutów, dziedziczenie) - klasy.

In [ ]:
from contextlib import contextmanager

# Podstawowy schemat
@contextmanager
def managed():
    print("enter")
    yield "resource"
    print("exit")


with managed() as r:
    print(f"using: {r}")

In [ ]:
import time
from contextlib import contextmanager

# @contextmanager z try/finally - gwarantowany teardown
@contextmanager
def timer():
    start = time.time()
    try:
        yield
    finally:
        elapsed = time.time() - start
        print(f"Elapsed: {elapsed:.4f}s")


with timer():
    total = sum(range(1_000_000))
    print(f"Sum: {total}")

---

### 🐍 Ćwiczenia - @contextmanager

**Ćw. 7.** Napisz `@contextmanager` o nazwie `log_block(name)`,
który wypisuje `"[name] start"` przy wejściu i `"[name] end"`
przy wyjściu z bloku.

**Ćw. 8.** Napisz `@contextmanager` o nazwie `open_write(path)`,
który otwiera plik do zapisu i zamyka go w `finally`. Obsłuż
`OSError` wypisując komunikat i nie propagując błędu.

**Ćw. 9. *(Trudniejsze)*** Napisz `@contextmanager` o nazwie
`temp_env(key, value)`, który tymczasowo ustawia zmienną środowiskową
(`os.environ[key] = value`), a po bloku przywraca pierwotną wartość
(lub usuwa, jeśli nie istniała).

In [ ]:
from contextlib import contextmanager

# Ćw. 7: log_block
@contextmanager
def log_block(name):
    ...


with log_block("data loading"):
    data = list(range(100))

# [data loading] start
# [data loading] end

In [ ]:
from contextlib import contextmanager

# Ćw. 8: open_write
@contextmanager
def open_write(path):
    f = None
    try:
        f = open(path, 'w', encoding='utf-8')
        yield f
    except OSError as e:
        print(f"Cannot open {path}: {e}")
    finally:
        ...


with open_write('test_write.txt') as f:
    f.write('hello context manager')

with open('/root/forbidden.txt') as f:   # OSError
    f.write('blocked')

In [ ]:
import os
from contextlib import contextmanager

# Ćw. 9: temp_env
@contextmanager
def temp_env(key, value):
    old = os.environ.get(key)   # None jeśli nie istnieje
    os.environ[key] = value
    try:
        yield
    finally:
        # hint: os.environ.pop(key) lub os.environ[key] = old
        ...


os.environ.pop('MY_VAR', None)   # upewnij sie ze nie istnieje

with temp_env('MY_VAR', 'test_value'):
    print(os.environ.get('MY_VAR'))   # test_value

print(os.environ.get('MY_VAR'))       # None

## 4. 🔹 Praktyczne wzorce

Menedżery kontekstu szczególnie przydają się przy zasobach, które
trzeba niezawodnie zamknąć - nawet gdy program ulegnie awarii.
Najczęstsze przypadki użycia:

| Zasób | Wzorzec |
|-------|---------|
| Plik | `with open(path) as f` |
| Baza danych SQLite | `with sqlite3.connect(db) as conn` |
| Blokada wątku | `with threading.Lock() as lock` |
| Sesja HTTP | `with requests.Session() as s` |

In [ ]:
import sqlite3

# sqlite3.connect() jako CM - auto commit/rollback
with sqlite3.connect(':memory:') as conn:
    conn.execute(
        'CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT, age INTEGER)'
    )
    conn.execute('INSERT INTO users (name, age) VALUES (?, ?)', ('Alice', 30))
    conn.execute('INSERT INTO users (name, age) VALUES (?, ?)', ('Bob', 25))

    cursor = conn.execute('SELECT * FROM users')
    for row in cursor:
        print(row)

In [ ]:
import sqlite3
from contextlib import contextmanager

# Wlasny CM do bazy danych z obsluга transakcji
class DatabaseManager:
    def __init__(self, db_path):
        self.db_path = db_path
        self.conn = None

    def __enter__(self):
        self.conn = sqlite3.connect(self.db_path)
        return self.conn

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is None:
            self.conn.commit()
        else:
            self.conn.rollback()
        self.conn.close()


with DatabaseManager(':memory:') as conn:
    conn.execute('CREATE TABLE items (name TEXT, qty INTEGER)')
    conn.execute('INSERT INTO items VALUES (?, ?)', ('laptop', 5))
    rows = conn.execute('SELECT * FROM items').fetchall()
    print(rows)   # [('laptop', 5)]

---

### 🐍 Ćwiczenia - Praktyczne wzorce

**Ćw. 10.** Napisz klasę `ReadOnlyFile(path)` - menedżer kontekstu,
który otwiera plik tylko do odczytu. Metoda `write()` powinna
zgłaszać `PermissionError`.

**Ćw. 11.** Używając `@contextmanager` napisz `managed_db(path)`,
który otwiera połączenie SQLite, `yield`-uje kursor, a po bloku
wykonuje `commit()` (lub `rollback()` przy wyjątku) i zamyka połączenie.

**Ćw. 12. *(Trudniejsze)*** Napisz klasę `APISession(base_url)`,
która w `__enter__` tworzy `requests.Session()` z nagłówkiem
`User-Agent`, udostępnia metodę `get(endpoint)` i w `__exit__`
zamyka sesję.

In [ ]:
# Ćw. 10: ReadOnlyFile
class ReadOnlyFile:
    def __init__(self, path):
        self.path = path
        self._file = None

    def __enter__(self):
        self._file = open(self.path, 'r', encoding='utf-8')
        return self

    def read(self):
        return self._file.read()

    def write(self, data):
        ...

    def __exit__(self, *args):
        self._file.close()


with ReadOnlyFile('greetings.txt') as f:
    print(f.read())
    try:
        f.write('blocked')   # PermissionError
    except PermissionError as e:
        print(e)

In [ ]:
import sqlite3
from contextlib import contextmanager

# Ćw. 11: managed_db
@contextmanager
def managed_db(path):
    conn = sqlite3.connect(path)
    cursor = conn.cursor()
    try:
        yield cursor
        ...
    except Exception:
        ...
        raise
    finally:
        conn.close()


with managed_db(':memory:') as cur:
    cur.execute('CREATE TABLE notes (text TEXT)')
    cur.execute('INSERT INTO notes VALUES (?)', ('hello',))
    print(cur.execute('SELECT * FROM notes').fetchall())

In [ ]:
import requests

# Ćw. 12: APISession
class APISession:
    def __init__(self, base_url):
        self.base_url = base_url
        self.session = None

    def __enter__(self):
        self.session = requests.Session()
        self.session.headers['User-Agent'] = 'MyApp/1.0'
        return self

    def get(self, endpoint):
        ...

    def __exit__(self, *args):
        ...


with APISession('https://jsonplaceholder.typicode.com') as api:
    data = api.get('/users/1')
    print(data['name'])   # Leanne Graham